# InSight Time & Motion Bench on a Colab GPU
**The/Nudge Institute · InSight × SIDBI chikankari pilot**

Runs the slower engines (official **TransRAC** and **RepNet**) on a free Colab GPU, using exactly the same code as the
laptop app, and saves results in the same format. Afterwards, add them to the laptop app on
*Results → Add results from another computer or Colab*.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU*.
2. In Google Drive, make a folder `InSight_TimeMotion` containing:
   - `InSight_TimeMotion_Bench.zip` (the app),
   - a `videos` folder with the videos to analyse,
   - the TransRAC checkpoint file (from the TransRAC GitHub page, OneDrive link, extraction code `transrac`),
   - optionally `manual_counts.csv` (the Validate page template) to analyse only the validation clips.

> **Privacy:** this puts artisan videos on Google Drive and Colab. Check that the consent form and the
> organisation's data policy allow it, use pseudonymous file names, and delete the videos from Drive when done.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: choose Runtime > Change runtime type > T4 GPU"
import sys; print(sys.version)

In [ ]:
# 2. Connect Google Drive and set the folders (edit these if your folder names differ)
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/InSight_TimeMotion'
APP_ZIP = f'{DRIVE}/InSight_TimeMotion_Bench.zip'
VIDEO_DIR = f'{DRIVE}/videos'
RESULTS_DIR = f'{DRIVE}/results_colab'          # results are written straight to Drive
TRANSRAC_CKPT = ''                               # leave blank to find the first .pt/.pth/.tar file in DRIVE
MANUAL_COUNTS = f'{DRIVE}/manual_counts.csv'     # optional

import glob
import os
if not TRANSRAC_CKPT:
    found = [p for p in glob.glob(f'{DRIVE}/*') if p.lower().endswith(('.pt', '.pth', '.tar', '.ckpt'))]
    TRANSRAC_CKPT = found[0] if found else ''
print('App zip:', os.path.exists(APP_ZIP), '| videos:', len(glob.glob(f'{VIDEO_DIR}/**/*.*', recursive=True)),
      '| TransRAC checkpoint:', TRANSRAC_CKPT or 'NOT FOUND')

In [ ]:
# 3. Unpack the app and install what it needs (PyTorch is already on Colab)
!rm -rf /content/app && mkdir -p /content/app && unzip -q "$APP_ZIP" -d /content/app
%cd /content/app/InSight_TimeMotion_Bench
!pip install -q timm einops mediapipe

In [ ]:
# 4. Download model files and add the TransRAC checkpoint
!python setup_models.py
import shutil
import os
if TRANSRAC_CKPT:
    os.makedirs('models/transrac', exist_ok=True)
    shutil.copy(TRANSRAC_CKPT, 'models/transrac/')
    print('TransRAC checkpoint copied.')

In [ ]:
# 5. Self-test (optional, ~2 minutes): checks every engine runs
!python self_test.py

In [ ]:
# 6a. Validation clips only (if you uploaded manual_counts.csv) - recommended first
import os
os.environ['INSIGHT_VIDEO_DIR'] = VIDEO_DIR
os.environ['INSIGHT_RESULTS_DIR'] = RESULTS_DIR
ENGINES = 'transrac,repnet_torch'
SETTINGS = ''   # e.g. '--setting transrac.window_sec=10 --setting repnet_torch.strides=2'
if os.path.exists(MANUAL_COUNTS):
    !python run_clips.py --clips "$MANUAL_COUNTS" --engines $ENGINES $SETTINGS
else:
    print('No manual_counts.csv found - skip to 6b.')

In [ ]:
# 6b. Whole videos (all videos in VIDEO_DIR; already-analysed ones are skipped)
#     Add --minutes 10 to analyse only the first 10 minutes of each video.
!python batch.py --engines $ENGINES $SETTINGS

## 7. Bring the results back to the laptop
The results are in Google Drive under `InSight_TimeMotion/results_colab/`:
- `master_results.csv` → on the laptop app, open **Results → Add results from another computer or Colab** and upload it.
  On the **Validate** page, choose these engines under *Also compare saved results for the same clips*.
- `runs/` → frame-level files for each analysis (optional).

**Window length for TransRAC.** TransRAC turns each window (20 s by default) into 64 frames. If a cycle spans fewer than
about four of those frames, the engine warns you; then re-run with shorter windows, e.g. add
`--setting transrac.window_sec=10` to the commands in step 6. Test a few lengths on the validation clips first.
Other settings work the same way, e.g. `--setting repnet_torch.strides=2` to fix RepNet's stride.